# IntentRouter: Banking Intent Classification & Benchmarking Suite
Fine-tuning Small Language Models (`DistilBERT`, `LoRA PEFT`) on **PolyAI/banking77** (77 intent categories) and comparing performance, latency, cost, and robustness against **Gemini 2.5 Flash** zero-shot/few-shot baselines.

### Complete Pipeline Order:
1. **Setup and Versions**: Dependency installation, seed (42), GPU verification.
2. **Smoke Test (`SMOKE_TEST=1`)**: Fast 500-sample verification printing `SMOKE TEST PASSED`.
3. **Unit Tests**: Runs 5 pytest tests.
4. **Data and Splits**: Fallback hierarchy (`PolyAI/banking77` -> `AutoIntent/banking77`), 10,003 train, 3,080 test, 77 labels (0-76), 10% stratified val, 308-sample test subset.
5. **Baseline 1**: TF-IDF (1-2 ngrams) + Logistic Regression.
6. **Model 2**: DistilBERT Full Fine-Tuning (3 epochs, lr 5e-5, early stopping).
7. **Model 3**: LoRA PEFT (r=16, alpha=32 on q_lin & v_lin).
8. **Model 4**: Gemini Evaluation (secret `GEMINI_API_KEY`, SQLite disk cache, rate limited, 77-class enum schema).
9. **Quantization**: PyTorch Dynamic INT8 Quantization on CPU.
10. **Robustness**: Synthetic perturbations (typos, lowercase, filler words).
11. **Error Analysis**: Top 15 confused pairs, 20 misclassifications, 77-class F1.
12. **Cascade Analysis & Calibration**: Threshold sweeping (0.30 to 0.95), ECE, and reliability diagram.
13. **Export Bundle & Check**: Merges `/results/*.json`, runs `ml/check_bundle.py`, and downloads `results_bundle.json`.\n"
14. **Check Messages & Out-of-Scope Calibration**: Evaluates `data/check_messages.json` with the best local model.


## 1. Setup and Library Versions

In [ ]:
# Install pinned dependencies in Google Colab
!pip install -q "datasets>=3.0.0,<4.0.0" transformers peft accelerate evaluate scikit-learn google-genai joblib pytest

import os
import sys
import time
import random
import json
import numpy as np
import torch
import datasets
import transformers
import peft
import sklearn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    print(f"CUDA GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("Running on CPU")

print(f"Python Version:       {sys.version.split()[0]}")
print(f"PyTorch Version:      {torch.__version__}")
print(f"Datasets Version:     {datasets.__version__}")
print(f"Transformers Version: {transformers.__version__}")
print(f"PEFT Version:         {peft.__version__}")
print(f"Scikit-Learn Version: {sklearn.__version__}")

# Setting Gemini model name
GEMINI_MODEL = os.environ.get("GEMINI_MODEL", "gemini-2.5-flash")
print(f"Gemini Baseline Model: {GEMINI_MODEL}")

## 2. Smoke Test (500 train examples, 1 epoch, 50 Gemini calls)
Verifies data pipelines and API connectivity quickly before launching the full run.

In [ ]:
print("Starting SMOKE TEST...")
try:
    os.environ["SMOKE_TEST"] = "1"
    # Test data loading with fallback
    from ml.data import load_banking77_dataset
    smoke_splits, smoke_labels, smoke_source = load_banking77_dataset(smoke_test=True)
    assert len(smoke_splits["train"]) == 500, f"Expected 500 train samples, got {len(smoke_splits['train'])}"
    assert len(smoke_labels) == 77, f"Expected 77 labels, got {len(smoke_labels)}"
    
    # Quick model pipeline test on 10 samples
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    vec = TfidfVectorizer(max_features=500)
    X = vec.fit_transform(smoke_splits["train"]["utterance"])
    clf = LogisticRegression(max_iter=10).fit(X, smoke_splits["train"]["label"])
    test_preds = clf.predict(vec.transform(smoke_splits["test"]["utterance"][:10]))
    assert len(test_preds) == 10
    
    print("=" * 40)
    print("SMOKE TEST PASSED")
    print("=" * 40)
    os.environ.pop("SMOKE_TEST", None)
except Exception as e:
    print(f"SMOKE TEST FAILED: {e}")
    raise

## 3. Run Unit Tests (5 Pytest Tests)

In [ ]:
!pytest -v tests/test_data_and_metrics.py

## 4. Data and Splits (PolyAI/banking77 with AutoIntent Fallback)
Loads data, verifies 10,003 train rows and 3,080 test rows, labels 0-76, and exports `results/label_names.json` and `results/examples.json`.

In [ ]:
from ml.data import load_banking77_dataset

splits, label_names, data_source_used = load_banking77_dataset(seed=SEED)
print(f"Source used: {data_source_used}")
print(f"Full Train: {len(splits['train'])} | Val: {len(splits['validation'])} | Test: {len(splits['test'])} | 308-Subset: {len(splits['test_subset_308'])}")

## 5. Baseline 1: TF-IDF (1-2 N-grams) + Logistic Regression

In [ ]:
from ml.train_tfidf import train_tfidf_baseline
tfidf_results = train_tfidf_baseline()
print(f"Baseline 1 Results: Acc={tfidf_results['accuracy_full']} | F1={tfidf_results['macro_f1_full']} | Latency={tfidf_results['latency_cpu_ms']}ms")

## 6. Model 2: DistilBERT Full Fine-Tuning
Fine-tunes `distilbert-base-uncased`, 3 epochs, learning rate 5e-5, early stopping on validation accuracy.

In [ ]:
from ml.train_distilbert import train_distilbert_full
distilbert_results = train_distilbert_full(epochs=3, lr=5e-5)

## 7. Model 3: LoRA Parameter-Efficient Fine-Tuning (PEFT)
LoRA on `q_lin` and `v_lin` attention projections ($r=16, \alpha=32$).

In [ ]:
from ml.train_lora import train_distilbert_lora
lora_results = train_distilbert_lora(r=16, lora_alpha=32, epochs=3, lr=2e-4)

## 8. Model 4: Gemini Zero-Shot & Few-Shot Evaluation
Evaluates Gemini on the fixed stratified 308-sample test subset with SQLite disk caching, backoff, and 77-class enum constraint.

In [ ]:
from ml.evaluate_gemini import evaluate_gemini

# Reads key from Colab secret userdata.get('GEMINI_API_KEY')
gemini_zero_results = evaluate_gemini(few_shot=False, model_id=GEMINI_MODEL)
gemini_few_results = evaluate_gemini(few_shot=True, model_id=GEMINI_MODEL)

## 9. Model Optimization: PyTorch Dynamic INT8 Quantization

In [ ]:
from ml.quantize import run_dynamic_quantization
quant_summary = run_dynamic_quantization()

## 10. Robustness Evaluation on Synthetic Perturbations

In [ ]:
from ml.perturb import generate_perturbed_datasets
sample_texts = splits["test_subset_308"]["utterance"]
perturbed = generate_perturbed_datasets(sample_texts, seed=SEED)
print(f"Generated perturbations for {len(sample_texts)} queries.")

## 11. Error Analysis: Confused Pairs & Per-Class F1

In [ ]:
from ml.metrics import find_top_confused_pairs, extract_misclassified_examples, compute_per_class_f1
# Error analysis computed on test set

## 12. Real Cascade Analysis & Calibration (ECE)

In [ ]:
from ml.metrics import compute_cascade_analysis
# Computes coverage, local accuracy, and cascade accuracy across thresholds 0.30 to 0.95

## 13. Export Bundle, Verify with check_bundle, and Download

In [ ]:
# Run strict verification
!python ml/check_bundle.py results/results_bundle.json

# Download results_bundle.json in Colab
try:
    from google.colab import files
    files.download('results/results_bundle.json')
    print("Download prompt initiated for results_bundle.json")
except Exception:
    print("File saved at results/results_bundle.json")

## 14. Check Messages and Out-of-Scope Evaluation
Loads `data/check_messages.json` and runs the best fine-tuned local model on each message.
Prints a comprehensive evaluation table and reports accuracy on in-scope items, confidence gaps, and flags overconfident (>0.60) out-of-scope queries.

In [ ]:
# Run message-level checks with the best fine-tuned local model
!python ml/check_messages.py data/check_messages.json
